# GPU加速されたDFTバッチ計算（HOMO/LUMO・構造最適化・TD-DFT）

このノートブックは、SMILESを含むデータセットを読み込み、RDKitを用いて3D構造を生成し、**GPU4PySCF** + **geomeTRIC**を用いてGPU上で構造最適化を行い、さらにHOMO/LUMO、双極子モーメント、TD-DFT励起状態（S1/T1/f）などの物性値を算出・抽出します。

[手順]
1. RDKitを使って3D構造作成
2. 計算のパラーメータ設定
3. 構造最適化
4. TD計算

In [1]:
# 1. ライブラリの読み込み

# 標準ライブラリ
import os      # 入出力ファイルのパス操作とディレクトリ作成
import sys     # PySCFのログがノートブック出力そのものかを判定する
import re      # 材料名をファイル名に使える文字へ置換
import time    # 分子ごとの計算時間を計測
import gc      # 計算オブジェクトを破棄し、GPUメモリを返せるようにする
import ctypes  # cuTENSORの共有ライブラリを先に読み込む

# 数値・表データ
import numpy as np   # 軌道番号や双極子モーメントなどの数値処理
import pandas as pd  # 入力CSVの読み込みと計算結果の書き出し

# 分子構造（SMILESから3次元座標を作る）
import rdkit                    # バージョン確認用
from rdkit import Chem          # SMILESの解釈、水素付加、形式電荷
from rdkit.Chem import AllChem  # 初期3次元構造の生成と力場による最適化

# GPU計算の土台
import cupy     # GPU上の配列演算。gpu4pyscfが内部で使う
import cutensor  # NVIDIA cuTENSOR本体。共有ライブラリの場所を得る
cutensor_dir = os.path.join(list(cutensor.__path__)[0], "lib")
ctypes.CDLL(os.path.join(cutensor_dir, "libcutensor.so.2"), mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(os.path.join(cutensor_dir, "libcutensorMg.so.2"), mode=ctypes.RTLD_GLOBAL)
from cupy_backends.cuda.libs import cutensor as cutensor_lib  # CuPyから使うcuTENSOR

# 量子化学計算（PySCF / GPU4PySCF）
import pyscf       # 量子化学計算の本体
import gpu4pyscf   # PySCFのGPU版。DFTとTD-DFTをGPUで実行する
import geometric   # 構造最適化エンジン geomeTRIC
from pyscf import gto  # 座標・基底関数・電荷から計算用の分子を定義
from pyscf.geomopt.geometric_solver import kernel as geom_kernel  # geomeTRIC。収束フラグも受け取る
from pyscf.tools import cubegen  # HOMO/LUMOをcubeファイルとして書き出す
from gpu4pyscf.dft import rks as dft_rks    # GPU上のDFT（基底状態のエネルギーと軌道）
from gpu4pyscf.tdscf import rks as td_rks  # GPU上のTD-DFT（励起状態 S1/T1）

print(f"RDKit バージョン: {rdkit.__version__}")
print(f"PySCF バージョン: {pyscf.__version__}")
print(f"GPU4PySCF バージョン: {gpu4pyscf.__version__}")
print(f"geomeTRIC バージョン: {geometric.__version__}")
print(f"CuPy バージョン: {cupy.__version__}")
print(f"cuTENSOR バージョン: {cutensor_lib.get_version()}")

def release_gpu():
    # 参照が切れた計算オブジェクトを回収し、CuPyが握ったままのVRAMをOSへ返す
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()

def close_pyscf_log(obj):
    # PySCFはログを開きっぱなしにする。次の計算で同じパスを使うと 'w' で中身が消える
    stdout = getattr(obj, "stdout", None)
    if stdout is None or stdout in (sys.stdout, sys.stderr):
        return
    try:
        stdout.flush()
    except Exception:
        pass
    try:
        if not stdout.closed:
            stdout.close()
    except Exception:
        pass


RDKit バージョン: 2026.03.3
PySCF バージョン: 2.13.1
GPU4PySCF バージョン: 1.7.6
geomeTRIC バージョン: 1.1.1
CuPy バージョン: 14.1.1
cuTENSOR バージョン: 20700


In [2]:
# 2. データセットの読み込み
csv_path = 'data/test_dft.csv'
# 結果を保存（および再開）するCSVファイルのパス
output_csv = 'output/test_dft_results.csv'

df = pd.read_csv(csv_path)
print(f"読み込まれた化合物数: {len(df)}")
df.head()

読み込まれた化合物数: 2


,material,SMILES
0,ICzCN,C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)...
1,ICzCYP,C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=...


In [3]:
# 3. SMILESから3D構造の生成を定義 (RDKit)
def smiles_to_3d_block(smiles):
    """
    SMILES文字列をPySCF用の3D座標ブロック文字列に変換します。
    戻り値: (座標文字列, 電荷) または 失敗時は (None, None)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    
    # 水素原子の付加
    mol = Chem.AddHs(mol)
    
    # 3D埋め込み（初期座標の生成）
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    if AllChem.EmbedMolecule(mol, params) == -1:
        # デフォルトの埋め込みに失敗した場合はランダム初期座標で再試行
        params.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, params) == -1:
            return None, None
            
    # 力場 (MMFF94) を用いた高速構造最適化を行い、妥当な初期構造を取得
    # MMFFが組めないときは例外ではなく -1 が返る。そのときだけUFFへ替える
    try:
        mmff_failed = AllChem.MMFFOptimizeMolecule(mol) == -1
    except Exception:
        mmff_failed = True
    if mmff_failed:
        try:
            AllChem.UFFOptimizeMolecule(mol)
        except Exception:
            pass  # 最適化されていない3D座標のまま処理を継続
            
    # 3D座標の抽出
    conf = mol.GetConformer()
    coords = []
    for i, atom in enumerate(mol.GetAtoms()):
        pos = conf.GetAtomPosition(i)
        symbol = atom.GetSymbol()
        coords.append(f"{symbol} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}")
        
    coord_string = "; ".join(coords)
    charge = Chem.GetFormalCharge(mol)
    
    return coord_string, charge


In [ ]:
# 4. Define DFT Structure Optimization and TD-DFT Function
def run_dft_optimization(coord_string, charge=0, xc='b3lyp', basis='6-31g', mol_name='unknown', out_dir='output', save_chk=True, exact_exchange=False):
    """
    PySCFおよびgeomeTRICを用いた構造最適化と、続くTD-DFT（S1, T1）計算を実行します。
    また、最適化構造のXYZファイル、HOMO/LUMOのCubeファイル、および詳細なLogファイルを出力します。

    流れは次のとおりです。
    1. 初期座標でDFTの計算条件を作る
    2. 最適化済みxyzがなければ構造最適化し、あればその座標を読む
    3. 最適化構造でSCFをやり直す（ここが電子状態の本番計算）
    4. HOMO/LUMO、Mulliken電荷、Molden、Cube、基底状態の双極子、TD-DFTを出す

    PySCFは output にファイルパスを渡すと、そのファイルを 'w' で開き直します。
    構造最適化と単点計算で同じパスを使うと最適化ログが消えるので、ファイルを分けます。
    構造最適化は {名前}_geomopt.log、単点計算以降は {名前}.log です。
    """
    # 単点計算・TD-DFTのログ。構造最適化のログとは別ファイルにする
    log_file = os.path.join(out_dir, f"{mol_name}.log")
    geom_log = os.path.join(out_dir, f"{mol_name}_geomopt.log")
    xyz_path = os.path.join(out_dir, f"{mol_name}_opt.xyz")
    skip_opt = os.path.exists(xyz_path)

    print("  -> 構造最適化 (DFT) を実行中...")
    # RDKit由来の座標文字列からPySCFの分子を作る。この行ではまだエネルギーは計算しない
    # verbose=4 は反復ログを詳しく残す指定。既存のxyzを読むときは、ここではログを出さない
    if skip_opt:
        mol = gto.M(atom=coord_string, basis=basis, charge=charge, verbose=0)
    else:
        mol = gto.M(atom=coord_string, basis=basis, charge=charge, verbose=4, output=geom_log)
    # 電子数が奇数の分子を閉殻RKSで回すと、HOMO/LUMOも励起状態も意味を持たない
    if mol.nelectron % 2 != 0:
        close_pyscf_log(mol)
        raise ValueError(
            f"電子数が奇数 ({mol.nelectron}) です。このノートブックは閉殻のRKS専用です。"
            "イオンやラジカルは dft_ip_reorg.ipynb / dft_ea_reorg.ipynb を使ってください。"
        )
    # exact_exchange=False: JもKもRI近似。True: JだけRI近似し、交換Kは4中心積分で厳密に計算する
    def build_rks(target_mol):
        rks = dft_rks.RKS(target_mol, xc=xc)
        if exact_exchange:
            return rks.density_fit(only_dfj=True)
        return rks.density_fit()

    if exact_exchange:
        print("  -> 2電子積分: Coulomb(J)はRI近似、交換(K)は厳密計算")
    else:
        print("  -> 2電子積分: Coulomb(J)と交換(K)の両方をRI近似")

    if skip_opt:
        print("  -> 既存の最適化構造が見つかりました。構造最適化をスキップします。")
        # xyz全文は、あとで関数の戻り値として返す
        with open(xyz_path, "r") as f_xyz:
            lines = f_xyz.readlines()
        opt_xyz_str = "".join(lines)

        # xyzの1行目は原子数、2行目はコメント。3行目以降の「元素 x y z」をPySCFの入力形式に戻す
        coord_list = []
        for line in lines[2:]:
            parts = line.strip().split()
            if len(parts) >= 4:
                coord_list.append(f"{parts[0]} {parts[1]} {parts[2]} {parts[3]}")
        # 例: "C 0.0 0.0 0.0; N 1.2 0.0 0.0"
        opt_coord_string = "; ".join(coord_list)
        if not opt_coord_string:
            raise ValueError(f"最適化構造のxyzを読めませんでした: {xyz_path}")
        close_pyscf_log(mol)
    else:
        mf = build_rks(mol)
        # 交換相関項の数値積分グリッド。3はPySCFの初期値で、GaussianのIntegral=Fineに近い
        # 比較表の数値は、これより粗い level=1 で計算している
        mf.grids.level = 3
        # geomeTRICが、DFTのエネルギーと原子核にかかる力を使って構造を動かす。上限は100ステップ
        # optimize() は収束フラグを捨てて分子だけ返す。未収束の構造を成功にしないため kernel を使う
        conv, mol_opt = geom_kernel(mf, maxsteps=100)
        if not conv:
            close_pyscf_log(mol)
            raise RuntimeError("構造最適化が100ステップ以内に収束しませんでした。maxsteps を増やすか、初期構造を確認してください。")

        # 最適化後の座標（Bohr）と元素記号を取り出す
        opt_coords = mol_opt.atom_coords()
        opt_symbols = [mol_opt.atom_symbol(i) for i in range(mol_opt.natm)]

        # 標準xyzの先頭2行は、原子数とタイトル
        xyz_lines = [f"{mol_opt.natm}", f"{mol_name} optimized by DFT"]
        coord_list = []
        for sym, pos in zip(opt_symbols, opt_coords):
            # PySCF内部はBohr。可視化も次の計算もAngstromに揃える（1 Bohr = 0.529177210903 Å）
            pos_ang = pos * 0.529177210903
            coord_list.append(f"{sym} {pos_ang[0]} {pos_ang[1]} {pos_ang[2]}")
            xyz_lines.append(f"{sym} {pos_ang[0]:.5f} {pos_ang[1]:.5f} {pos_ang[2]:.5f}")

        opt_coord_string = "; ".join(coord_list)  # この後の電子状態計算へ渡す座標
        opt_xyz_str = "\n".join(xyz_lines) + "\n"

        with open(xyz_path, "w") as f_xyz:
            f_xyz.write(opt_xyz_str)
        # 構造を動かしていたオブジェクトは、単点計算では使わない
        del mol_opt
        # 最適化ログを閉じてから単点計算に入る。同じファイルを使い回すと中身が消える
        close_pyscf_log(mol)
        del mf
        release_gpu()

    # 最適化そのものではなく、最適化後の座標で分子を作り直してSCFを収束させる
    # 既存xyzを読んだ場合は、ここで初めて電子状態計算を行う
    print("  -> 最適化構造で電子状態計算を実行中...")
    mol = gto.M(atom=opt_coord_string, basis=basis, charge=charge, verbose=4, output=log_file)
    mf_opt = build_rks(mol)
    if save_chk:
        # 収束した軌道をバイナリで残す。後段のMolden変換で読む
        mf_opt.chkfile = os.path.join(out_dir, f"{mol_name}.chk")
    mf_opt.grids.level = 3  # 構造最適化と同じ密度にする。ここだけ変えると、最適化と物性で格子がずれる
    mf_opt.kernel()  # SCFを回し、軌道エネルギー、占有数、全エネルギーを確定する

    # 軌道エネルギーはHartreeで得られるので、eVへ換算する
    HARTREE_TO_EV = 27.211386
    mo_energy = mf_opt.mo_energy.get()  # GPU上の配列をCPUのNumPy配列へコピーする
    mo_occ = mf_opt.mo_occ.get()        # 各軌道の電子占有数。閉殻の占有軌道は2、空軌道は0

    occupied_indices = np.where(mo_occ > 0)[0]    # 電子が入っている軌道の番号
    unoccupied_indices = np.where(mo_occ == 0)[0]  # 空軌道の番号

    if len(occupied_indices) == 0 or len(unoccupied_indices) == 0:
        raise ValueError("分子軌道を正しく識別できませんでした。")

    homo_idx = np.max(occupied_indices)    # 占有軌道で最も高いエネルギー = HOMO
    lumo_idx = np.min(unoccupied_indices)  # 空軌道で最も低いエネルギー = LUMO
    homo_ev = mo_energy[homo_idx] * HARTREE_TO_EV
    lumo_ev = mo_energy[lumo_idx] * HARTREE_TO_EV

    print("  -> Mulliken電荷を計算中...")
    mf_cpu = mf_opt.to_cpu()  # Mulliken解析はCPU版PySCFの関数なので、計算結果をCPUへ移す
    pop, chg = mf_cpu.mulliken_pop()  # popは軌道ごとの電子数、chgは原子ごとの電荷。画面とログに出る

    # チェックポイントから、軌道可視化用のMoldenファイルを作る
    from pyscf.tools import molden
    molden_path = os.path.join(out_dir, f"{mol_name}.molden")
    chk_path = os.path.join(out_dir, f"{mol_name}.chk")
    if os.path.exists(chk_path):
        print(f"  -> Moldenファイルを出力中: {molden_path}")
        molden.from_chkfile(molden_path, chk_path)

    print("  -> HOMO / LUMO のCubeファイルを生成中...")
    mo_coeff_cpu = mf_opt.mo_coeff.get()  # 分子軌道係数。Cube生成はCPU側で行う
    # 指定した軌道の広がりを3次元格子の値として保存する。Avogadroなどで等値面にできる
    cubegen.orbital(mol, os.path.join(out_dir, f"{mol_name}_HOMO.cube"), mo_coeff_cpu[:, homo_idx])
    cubegen.orbital(mol, os.path.join(out_dir, f"{mol_name}_LUMO.cube"), mo_coeff_cpu[:, lumo_idx])

    print("  -> 基底状態の双極子モーメントを計算中...")
    dip_mom = mf_opt.dip_moment(unit='Debye')  # 励起状態ではなく、SCF後の基底状態の双極子ベクトル (x, y, z)
    dipole_val = np.linalg.norm(dip_mom)  # ベクトルの長さ。向きを除いた大きさ

    print("  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...")
    td_s = td_rks.TDDFT(mf_opt)  # 収束済みの基底状態軌道から、一重項の励起状態を求める
    td_s.nstates = 3  # 低い方から3状態を解く。使うのはそのうち一番低いS1
    td_s.kernel()
    s1_ev = td_s.e[0] * HARTREE_TO_EV  # S1の励起エネルギー
    f_s1 = td_s.oscillator_strength()[0]  # S1の振動子強度。吸収の強さの目安
    # 一重項の中間配列を捨ててから三重項に入る
    del td_s
    release_gpu()

    td_t = td_rks.TDDFT(mf_opt)
    td_t.singlet = False  # 三重項の励起状態に切り替える
    td_t.nstates = 3
    td_t.kernel()
    t1_ev = td_t.e[0] * HARTREE_TO_EV  # T1の励起エネルギー

    delta_est_ev = s1_ev - t1_ev  # 一重項と三重項の差 ΔEst
    e_tot = float(mf_opt.e_tot)  # 全エネルギーは数だけ残し、GPU上の計算オブジェクトは捨てる
    close_pyscf_log(mol)
    del mf_opt, td_t, mol
    release_gpu()
    # 戻り値: HOMO, LUMO, S1, T1, ΔEst, 振動子強度, 双極子の大きさ, 全エネルギー(Hartree), 最適化xyz
    return homo_ev, lumo_ev, s1_ev, t1_ev, delta_est_ev, f_s1, dipole_val, e_tot, opt_xyz_str


In [5]:
# 5. Batch process the dataset
XC_FUNCTIONAL = 'b3lyp'

# --- 利用可能な主な基底関数（PySCFでの指定名） ---
# 'sto-3g'    : 最小基底（計算は非常に早いが精度は低い）
# '6-31g'     : 一般的なダブルゼータ基底
# '6-31g*'    : 分極関数を重原子に追加（6-31G(d)と同義。標準的）
# '6-31+g*'   : 重原子に分散関数(+)と分極関数(*)を追加（6-31+G(d)と同義。アニオンやESP計算に最低限必要）
# '6-311+g**' : トリプルゼータ＋重原子に分散関数＋全原子に分極関数（高精度だが非常に重い）
# '6-31g**'   : 水素にも分極関数を追加（6-31G(d,p) と同義）
# '6-311g**'  : トリプルゼータ基底（6-311G(d,p)と同義。高精度だが計算が重い）
# 'def2-svp'  : DFT計算でよく推奨されるAhlrichs基底系

# --- DFT-D (分散力補正) について ---
# B3LYPなどの汎関数名に '-d3bj' を付けることで、Grimmeの分散力補正 (D3(BJ)) が適用されます。
# 分散力補正を入れることで、スタッキングなどの非共有結合性相互作用を精度良く見積もることができます。

BASIS_SET = '6-31g*'

# False: JとKの両方をRI近似する（現行の速い設定）
# True : Coulomb(J)だけRI近似し、混成汎関数で効く交換(K)は厳密に計算する
EXACT_EXCHANGE = False

# レジューム（再開）処理のハンドリング
if os.path.exists(output_csv):
    df_results = pd.read_csv(output_csv)
    print(f"既存の結果CSV（{output_csv}）を読み込んで計算を再開します。")
else:
    # オリジナルのデータセットをロードします（存在しない場合はエラー）
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"入力用CSVファイルが見つかりません: {csv_path}")
    df_results = pd.read_csv(csv_path)
    print(f"オリジナルのデータセット（{csv_path}）を読み込みました。")

def resolve_input_columns(df):
    # READMEどおり、材料IDは material / index、構造は SMILES という列名で拾う
    # 列名が無いCSVだけ、先頭2列にフォールバックする
    lookup = {}
    for col in df.columns:
        key = str(col).strip().lstrip("\ufeff").lower()
        lookup.setdefault(key, col)
    smiles_col = lookup.get("smiles")
    id_col = None
    for key in ("material", "index"):
        if key in lookup and lookup[key] != smiles_col:
            id_col = lookup[key]
            break
    if smiles_col is not None and id_col is not None:
        print(f"入力データ設定: '{id_col}' を材料ID、'{smiles_col}' をSMILESとして認識しました。")
        return id_col, smiles_col
    if len(df.columns) >= 2:
        id_col = df.columns[0]
        smiles_col = df.columns[1]
        print(
            "列名 material/index と SMILES が見つからないため、"
            f"1列目 '{id_col}' を材料ID、2列目 '{smiles_col}' をSMILESとして使います。"
        )
        return id_col, smiles_col
    raise ValueError("CSVには材料ID（material または index）と SMILES の2列が必要です。")

id_col, smiles_col = resolve_input_columns(df_results)
# プログラム内で統一的に扱うため、カラム名を 'Material' と 'SMILES' に変更します
df_results.rename(columns={id_col: 'Material', smiles_col: 'SMILES'}, inplace=True)

# 必要な結果出力用カラムが存在しない場合は初期化
target_cols = {
    'HOMO_eV': np.nan,
    'LUMO_eV': np.nan,
    'S1_eV': np.nan,
    'T1_eV': np.nan,
    'delta_Est_eV': np.nan,
    'f': np.nan,
    'dipole_Debye': np.nan,
    'Total_Energy_Hartree': np.nan,
    'elapsed_sec': np.nan,
    'state': 'pending'
}
for col, default_val in target_cols.items():
    if col not in df_results.columns:
        df_results[col] = default_val

OUT_DIR = 'output'
os.makedirs(OUT_DIR, exist_ok=True)


オリジナルのデータセット（data/test_dft.csv）を読み込みました。
入力データ設定: 1列目 'material' を材料ID、2列目 'SMILES' をSMILESとして認識しました。


In [6]:
print("=== バッチDFT計算を開始します ===")
for index, row in df_results.iterrows():
    name = row['Material']
    smiles = row['SMILES']
    safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', str(name))
    state = row['state']
    
    # 計算完了または失敗済みのデータはスキップ（レジューム機能）
    if state in ['completed', 'fail']:
        print(f"\n[{index + 1}/{len(df_results)}] スキップします: {name} (SMILES: {smiles}) - すでにステータスが {state} です")
        continue
        
    print(f"\n[{index + 1}/{len(df_results)}] 計算中: {name} (SMILES: {smiles})")
    df_results.at[index, 'state'] = 'running'
    df_results.to_csv(output_csv, index=False)

    
    mol_out_dir = os.path.join(OUT_DIR, safe_name)
    os.makedirs(mol_out_dir, exist_ok=True)
    
    # ステップ 1: SMILESから3D座標を生成
    try:
        coord_string, charge = smiles_to_3d_block(smiles)
        if coord_string is None:
            raise ValueError("RDKitによる3D構造生成に失敗しました。")
    except Exception as e:
        print(f"  [エラー] 3D構造の生成に失敗しました: {e}")
        df_results.at[index, 'state'] = 'fail'
        df_results.to_csv(output_csv, index=False)
        continue
        
    # ステップ 2: DFT構造最適化およびTD-DFT計算
    t0 = time.time()
    try:
        homo, lumo, s1, t1, delta_est_ev, f, dipole, energy, opt_coords = run_dft_optimization(
            coord_string, charge=charge, xc=XC_FUNCTIONAL, basis=BASIS_SET, mol_name=safe_name, out_dir=mol_out_dir, exact_exchange=EXACT_EXCHANGE
        )
        elapsed = time.time() - t0
        delta_est = s1 - t1  # ΔEst の計算
        print(f"  [成功] 計算完了 ({elapsed:.1f}秒). S1: {s1:.3f} eV, T1: {t1:.3f} eV, ΔEst: {delta_est:.3f} eV, f: {f:.4f}, dipole: {dipole:.3f} D")
        
        df_results.at[index, 'HOMO_eV'] = homo
        df_results.at[index, 'LUMO_eV'] = lumo
        df_results.at[index, 'S1_eV'] = s1
        df_results.at[index, 'T1_eV'] = t1
        df_results.at[index, 'delta_Est_eV'] = delta_est
        df_results.at[index, 'f'] = f
        df_results.at[index, 'dipole_Debye'] = dipole
        df_results.at[index, 'Total_Energy_Hartree'] = energy
        df_results.at[index, 'elapsed_sec'] = elapsed
        df_results.at[index, 'state'] = 'completed'
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  [エラー] DFT計算が異常終了しました ({elapsed:.1f}秒): {e}")
        df_results.at[index, 'elapsed_sec'] = elapsed
        df_results.at[index, 'state'] = 'fail'

    # 成功しても失敗しても、次の分子に入る前にVRAMを返す
    release_gpu()

    # 1材料の処理が終わるごとに、結果を逐次CSVに保存
    df_results.to_csv(output_csv, index=False)
    print(f"  結果を一時保存しました: {output_csv}")


=== バッチDFT計算を開始します ===

[1/2] 計算中: ICzCN (SMILES: C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)C7=CC=CC=C7)=CC=C(C1)C8=C(C(C=9C=CC(N%10C=%11C(C%12=CC=C%13N(C=%14C(C%13=C%10%12)=CC=CC%14)C=%15C=CC=CC%15)=CC=CC%11)=CC9)=CN=C8)C#N)


  -> 構造最適化 (DFT) を実行中...
output file: output/ICzCN/ICzCN.log


geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3c0596efb47fbb22e284659ce9197ef0d8c9f20d8.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%

  -> 2電子積分: Coulomb(J)と交換(K)の両方をRI近似


336 internal coordinates being used (instead of 336 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-32
Distance 1-73
Distance 2-3
Distance 2-29
Distance 3-4
Distance 3-22
Distance 4-5
Distance 4-9
Distance 5-6
Distance 5-74
Distance 6-7
Distance 6-75
Distance 7-8
Distance 7-76
Distance 8-9
Distance 8-77
Distance 9-10
Distance 10-11
Distance 10-22
Distance 11-12
Distance 11-78
Distance 12-13
Distance 12-79
Distance 13-14
Distance 13-21
Distance 14-15
Distance 14-23
Distance 15-16
Distance 15-20
Distance 16-17
Distance 16-80
Distance 17-18
Distance 17-81
Distance 18-19
Distance 18-82
Distance 19-20
Distance 19-83
Distance 20-21
Distance 21-22
Distance 23-24
Distance 23-28
Distance 24-25
Distance 24-84
Distance 25-26
Distance 25-85
Distance 26-27
Distance 26-86
Distance 27-28
Distance 27-87
Distance 28-88
Distance 29-30
Distance 29-89
Distance 30-31
Distance 30-90
Distance 31-32
Distance 31-33
Distance 32-91
Distance 33-34
Distance 33-70
Distance 34

  -> 最適化構造で電子状態計算を実行中...
overwrite output file: output/ICzCN/ICzCN.log
  -> Mulliken電荷を計算中...
  -> Moldenファイルを出力中: output/ICzCN/ICzCN.molden
  -> HOMO / LUMO のCubeファイルを生成中...
  -> 基底状態の双極子モーメントを計算中...
  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...
  [成功] 計算完了 (26564.9秒). S1: 2.577 eV, T1: 2.562 eV, ΔEst: 0.015 eV, f: 0.0138, dipole: 3.723 D
  結果を一時保存しました: output/test_dft_results.csv

[2/2] 計算中: ICzCYP (SMILES: C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=CC2)=CC=CC5)C=7C=CC(=CC7)C8=C(C=CC(=N8)C=9C=CC(N%10C=%11C(=CC=CC%11)C=%12C%10=C%13C%14=CC=CC=C%14N(C%13=CC%12)C=%15C=CC=CC%15)=CC9)C#N)


geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3c0596efb47fbb22e284659ce9197ef0d8c9f20d8.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%

  -> 構造最適化 (DFT) を実行中...
output file: output/ICzCYP/ICzCYP.log
  -> 2電子積分: Coulomb(J)と交換(K)の両方をRI近似


336 internal coordinates being used (instead of 336 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-9
Distance 1-22
Distance 2-3
Distance 2-7
Distance 3-4
Distance 3-8
Distance 4-5
Distance 4-73
Distance 5-6
Distance 5-74
Distance 6-7
Distance 6-75
Distance 7-76
Distance 8-9
Distance 8-27
Distance 9-10
Distance 10-11
Distance 10-20
Distance 11-12
Distance 11-26
Distance 12-13
Distance 12-23
Distance 13-14
Distance 13-20
Distance 14-15
Distance 14-19
Distance 15-16
Distance 15-77
Distance 16-17
Distance 16-78
Distance 17-18
Distance 17-79
Distance 18-19
Distance 18-80
Distance 19-81
Distance 20-21
Distance 21-22
Distance 21-82
Distance 22-83
Distance 23-24
Distance 23-84
Distance 24-25
Distance 24-85
Distance 25-26
Distance 25-86
Distance 26-87
Distance 27-28
Distance 27-32
Distance 28-29
Distance 28-88
Distance 29-30
Distance 29-89
Distance 30-31
Distance 30-33
Distance 31-32
Distance 31-90
Distance 32-91
Distance 33-34
Distance 33-38
Distance 34

  -> 最適化構造で電子状態計算を実行中...
overwrite output file: output/ICzCYP/ICzCYP.log
  -> Mulliken電荷を計算中...
  -> Moldenファイルを出力中: output/ICzCYP/ICzCYP.molden
  -> HOMO / LUMO のCubeファイルを生成中...
  -> 基底状態の双極子モーメントを計算中...
  -> TD-DFTによる一重項/三重項の計算を実行中 (S1, T1)...
  [成功] 計算完了 (27097.2秒). S1: 2.509 eV, T1: 2.462 eV, ΔEst: 0.047 eV, f: 0.0074, dipole: 4.327 D
  結果を一時保存しました: output/test_dft_results.csv


In [7]:
# 6. Save results
print("バッチ処理が完了しました。最終結果の概要は以下の通りです:")
df_results.head()

バッチ処理が完了しました。最終結果の概要は以下の通りです:


,Material,SMILES,HOMO_eV,LUMO_eV,S1_eV,T1_eV,delta_Est_eV,f,dipole_Debye,Total_Energy_Hartree,elapsed_sec,state
0,ICzCN,C=1C(N2C3=CC=CC=C3C4=CC=C5N(C6=CC=CC=C6C5=C24)...,-5.074328,-2.153250,2.577154,2.562201,0.014953,0.013783,3.722709,-2867.650493,26564.888154,completed
1,ICzCYP,C1=2C=3C(=CC=CC3)N(C1=C4C=5C(N(C=6C=CC=CC6)C4=...,-5.006275,-2.167158,2.508529,2.461933,0.046596,0.007401,4.327281,-2867.660890,27097.239512,completed


In [8]:
# 6. failした材料の抽出と保存
if os.path.exists(output_csv):
    df_final = pd.read_csv(output_csv)
    df_fail = df_final[df_final['state'] == 'fail']
    
    if len(df_fail) > 0:
        base_name, ext = os.path.splitext(output_csv)
        fail_csv = f"{base_name}_fail{ext}"
        df_fail.to_csv(fail_csv, index=False)
        print(f"failした材料（{len(df_fail)}件）を抽出して保存しました: {fail_csv}")
    else:
        print("failした材料はありませんでした。おめでとうございます！")
else:
    print(f"{output_csv} が見つかりません。")

failした材料はありませんでした。おめでとうございます！
